# ENEC 2026 · Day 3 · Skills, Progressive Disclosure, Memory and Governed Agents

**Time:** about 75 minutes  |  **Runs in:** Google Colab (free tier is fine)  |  **Needs:** OpenAI key in Colab Secrets (`OPENAI_API_KEY`)

**Data:** a small slice of the synthetic *Nuclear Enterprise 360* dataset, focused on **Asset A-001** (a cooling-water pump). All data is synthetic training data.

**Goal:** Give the agent reusable skills it loads only when needed, a small memory, an audit trail and a human-approval gate. Then compare a single agent with a supervisor and specialists.

> Training notice: nothing in this notebook is an engineering diagnosis or a maintenance authorisation. The AI supports a qualified human reviewer.

In [ ]:
# Install what Colab does not already have (about 20 seconds)
%pip install -q -U openai tiktoken

In [ ]:
# Get the course files (data, documents, skills) and the helper module.
# In Colab this clones the public course repo into /content/enec2026oct.
import os, sys, subprocess

REPO_URL = "https://github.com/Decoding-Data-Science/enec2026oct"
IN_COLAB = os.path.isdir("/content") and "google.colab" in sys.modules
ROOT = "/content/enec2026oct" if IN_COLAB else os.getcwd()

if IN_COLAB and not os.path.isdir(ROOT):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, ROOT], capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            "Could not clone the course repo:\n" + r.stderr[-300:] +
            "\nFallback: upload enec2026oct.zip in the Files panel (left sidebar), "
            "run  !unzip -q enec2026oct.zip -d /content  and run this cell again."
        )

sys.path.insert(0, os.path.join(ROOT, "colab"))
import enec_colab as enec
print("Course files ready at:", enec.ROOT)

In [ ]:
# Connect to OpenAI. The key comes from Colab Secrets and is never printed.
client = enec.get_client()
MODEL = enec.pick_model(client)     # first available of: gpt-5.4-mini, gpt-5-mini, gpt-4o-mini

## Six ideas to keep separate

| Layer | Meaning |
|---|---|
| **Tools** | What the agent can call (functions) |
| **Skills** | Reusable instructions for a type of task (a playbook) |
| **Progressive disclosure** | Load the short catalog first, the detailed skill only when selected |
| **Memory** | Facts carried between turns or sessions |
| **Governance** | Permissions, audit trail, approvals |
| **Multi-agent** | Several specialist agents coordinated by a supervisor |

In [ ]:
import json, pandas as pd, tiktoken, datetime as dt
enc = tiktoken.get_encoding("o200k_base")

tables = enec.load_tables()
con    = enec.make_db(tables)
docs   = enec.load_documents()
index  = enec.RAGIndex.build(client, docs)
hourly = tables["a001_sensor_hourly_90d"]
TOOLS, IMPLS = enec.build_a001_tools(con, index, hourly)
print([t["function"]["name"] for t in TOOLS])

## 1. Skills and progressive disclosure

A **skill** is a markdown playbook. The agent first sees only a tiny catalog, then loads the full skill it needs. This keeps prompts short and behaviour consistent.

In [ ]:
catalog = enec.load_skill_catalog()
print(catalog)

In [ ]:
print(enec.load_skill("reliability_review"))

In [ ]:
# How much do we save by NOT loading every skill into every prompt?
names = ["asset_summary", "approved_procedure", "reliability_review"]
all_tokens = sum(len(enc.encode(enec.load_skill(n))) for n in names)
catalog_tokens = len(enc.encode(catalog))
print(f"catalog only: {catalog_tokens} tokens | all full skills: {all_tokens} tokens")
print("One selected skill adds about", len(enc.encode(enec.load_skill('reliability_review'))), "tokens")

### Step A: the model picks a skill from the catalog (it sees only the catalog)
### Step B: we load that one skill and allow only the tools that skill permits

In [ ]:
ALLOWED = {"asset_summary": ["sql_query"],
           "approved_procedure": ["search_documents"],
           "reliability_review": ["sql_query", "search_documents", "forecast"]}

def choose_skill(question):
    system = ("Choose ONE skill from the catalog for the question. "
              'Return JSON: {"skill": "<name>", "why": "<short reason>"}\n\n' + catalog)
    return json.loads(enec.ask(client, question, system=system, json_mode=True))

def skill_agent(question):
    pick = choose_skill(question)
    skill = pick.get("skill") if pick.get("skill") in ALLOWED else "reliability_review"
    tools = [t for t in TOOLS if t["function"]["name"] in ALLOWED[skill]]
    system = (f"You are the A-001 assistant. Follow this skill exactly.\n\n{enec.load_skill(skill)}\n\n"
              f"{enec.SCHEMA_HINT}\nYou never authorise work; a qualified human decides.")
    final, trace, _ = enec.run_agent(client, question, system, tools, IMPLS, verbose=False)
    return skill, [t["tool"] for t in trace], final

for q in ["What is the health score of A-001?",
          "Which procedure is currently approved for A-001?",
          "Should A-001 be escalated for review?"]:
    skill, used, final = skill_agent(q)
    print(f"Q: {q}\n  skill = {skill} | tools used = {used}\n  {final[:600]}\n")

**Notice:** the `approved_procedure` skill is only allowed to search documents. Even if the model *wanted* to run SQL, the tool is not offered. That is permissioning by design.

## 2. Memory

Without memory every turn starts blank. A simple, transparent memory is a small dictionary of **facts we choose to keep**, saved to a file and injected into the system prompt.

Rules we follow: keep only what is useful, never store secrets or personal data, let the user see and clear it.

In [ ]:
import os
MEM_PATH = "/tmp/enec_agent_memory.json"
memory = json.load(open(MEM_PATH)) if os.path.exists(MEM_PATH) else {}

def remember(key, value):
    memory[key] = value
    json.dump(memory, open(MEM_PATH, "w"), indent=2)

def memory_block():
    return "MEMORY (facts the user asked us to keep):\n" + (json.dumps(memory, indent=2) if memory else "(empty)")

remember("focus_asset", "A-001")
remember("report_style", "short bullets, cite document IDs")
memory

In [ ]:
def chat_with_memory(question):
    system = ("You are the A-001 assistant. Use the MEMORY for the user's preferences. "
              "You never authorise work.\n" + memory_block() + "\n" + enec.SCHEMA_HINT)
    final, trace, _ = enec.run_agent(client, question, system, TOOLS, IMPLS, verbose=False)
    return final

print(chat_with_memory("Give me the current open high-priority work for our focus asset."))

## 3. Governance: audit trail and a human-approval gate

Two controls every enterprise agent needs:

1. **Audit trail:** record every tool call (who, what, when, result summary).
2. **Approval gate:** the agent may *prepare* a recommendation, but a consequential step goes into a **review queue** for a human. It cannot approve itself.

In [ ]:
AUDIT, REVIEW_QUEUE = [], []

def audited(name, fn):
    def wrapper(**kwargs):
        out = fn(**kwargs)
        AUDIT.append({"time": dt.datetime.now().strftime("%H:%M:%S"), "tool": name,
                      "args": json.dumps(kwargs)[:120], "result_chars": len(str(out))})
        return out
    return wrapper

def submit_for_human_review(summary: str, evidence_ids: str) -> str:
    """The agent's only 'action': put a briefing in the queue. It does NOT change anything."""
    REVIEW_QUEUE.append({"status": "PENDING_HUMAN_REVIEW", "summary": summary, "evidence": evidence_ids})
    return "Queued for qualified human review. No operational action has been taken."

GOV_TOOLS = TOOLS + [enec.tool_spec("submit_for_human_review",
    "Submit a short recommendation and the evidence IDs to the human review queue. This is the ONLY way to propose follow-up.",
    {"summary": {"type": "string"}, "evidence_ids": {"type": "string", "description": "Comma-separated document IDs / table names"}})]
GOV_IMPLS = {**{k: audited(k, v) for k, v in IMPLS.items()}, "submit_for_human_review": audited("submit_for_human_review", submit_for_human_review)}

system = ("You are the A-001 assistant. Investigate with the tools, then call submit_for_human_review exactly once with a short "
          "recommendation and the evidence IDs. You never authorise work.\n" + enec.SCHEMA_HINT)
final, trace, _ = enec.run_agent(client, "Prepare A-001 for human reliability review.", system, GOV_TOOLS, GOV_IMPLS, verbose=False)
print(final[:700])

In [ ]:
print("AUDIT TRAIL"); display(pd.DataFrame(AUDIT))
print("REVIEW QUEUE"); display(pd.DataFrame(REVIEW_QUEUE))

## 4. Single agent or supervisor + specialists?

A **supervisor** splits the work: a *data specialist* (SQL + forecast), a *document specialist* (search), and a *reviewer* that writes the briefing from their outputs. Each specialist has a smaller toolset and a narrower prompt.

In [ ]:
def specialist(name, role, tool_names, task):
    tools = [t for t in TOOLS if t["function"]["name"] in tool_names]
    system = f"You are the {name}. {role} Return concise findings only. You never authorise work.\n{enec.SCHEMA_HINT}"
    final, trace, _ = enec.run_agent(client, task, system, tools, IMPLS, verbose=False)
    return final, len(trace)

task = "Prepare A-001 for human reliability review."
data_out, n1 = specialist("data specialist", "You find structured facts and the vibration forecast.", ["sql_query", "forecast"],
                          "Collect A-001 health, open work orders, recent inspections and the 7-day vibration forecast.")
doc_out, n2 = specialist("document specialist", "You find the current approved guidance and flag DRAFT or SUPERSEDED material.", ["search_documents"],
                         "Find the approved inspection procedure, the escalation policy and the latest field/condition reports for A-001.")

reviewer = ("You are the reviewer. Write a 6-bullet briefing for a qualified human reviewer from the two reports. "
            "Keep database, forecast and document evidence separate. State gaps. Do not authorise work.")
brief = enec.ask(client, f"DATA SPECIALIST:\n{data_out}\n\nDOCUMENT SPECIALIST:\n{doc_out}", system=reviewer)
print(brief)
print(f"\nTool calls: data specialist {n1}, document specialist {n2}")

**When is multi-agent worth it?**

| Use multi-agent when | Stay with one agent when |
|---|---|
| Tasks need clearly different tools or permissions | The task fits one prompt and a few tools |
| You want to test each specialist separately | Latency and cost matter most |
| Different teams own different capabilities | You cannot yet explain why a second agent helps |

More agents means more calls, more latency and more places to fail. Start with one agent and split only when you can name the reason.

## Takeaways

- **Skills** make behaviour reusable; **progressive disclosure** keeps prompts small.
- **Memory** should be small, explicit and user-controlled.
- **Governance** is code: allowed tools per skill, an audit trail, and a human approval gate.
- Multi-agent is an architecture choice with a price, not an upgrade.

## Exercise

1. Write a new skill file `skills/work_order_triage.md` that is allowed to use SQL only, then add it to `ALLOWED` and the catalog text. Does `choose_skill` pick it?
2. Make `submit_for_human_review` reject a summary that contains no evidence IDs.
3. Add a `clear_memory()` function and show the user what is stored before clearing.